# Notebook 1 — XML Understanding & Validation

**Run this first.** Then run **02_bronze_ingest_xml.ipynb** with the same `xml_path` and `row_tag`.

In [0]:
dbutils.widgets.text(
    "xml_path",
    "/Volumes/dev_automotive/landing/landing_raw/final_geografic.xml",
    "XML file path (volume)",
)
dbutils.widgets.text("row_tag", "record", "rowTag (e.g. record) — use in Notebook 2")
xml_path = dbutils.widgets.get("xml_path").strip()
row_tag = dbutils.widgets.get("row_tag").strip()

# Always print these first so you have them even if later cells fail (e.g. malformed XML)
print(f"xml_path: {xml_path}")
print(f"row_tag:  {row_tag}")

## 1. Read file as text — confirm exists and not empty

In [0]:
df_text = spark.read.text(xml_path)
line_count = df_text.count()

assert line_count > 0, "❌ XML file is empty or not readable by Spark"
print(f"✅ File readable by Spark. Line count: {line_count:,}")

## 2. Structure & schema discovery (Spark XML)

In [0]:
sample_df = (
    spark.read
    .format("xml")
    .option("rowTag", row_tag)
    .option("inferSchema", True)
    .load(xml_path)
)

# When XML is malformed, Spark may infer only _corrupt_record — avoid .show() then (Spark disallows querying only that column)
cols = sample_df.columns
only_corrupt = cols == ["_corrupt_record"] or (len(cols) == 1 and "_corrupt_record" in cols)
if only_corrupt:
    print("⚠️ Malformed XML: Spark inferred only _corrupt_record. Use Notebook 2 with Run healer = true.")
    sample_df.printSchema()
else:
    assert sample_df.count() > 0, f"rowTag '{row_tag}' not found"
    sample_df.printSchema()
    sample_df.limit(10).show(truncate=False)

## 3. Structural validation (strict XML) — binary: valid / invalid

In [0]:
import xml.etree.ElementTree as ET

def is_valid_xml(file_path: str) -> bool:
    try:
        ET.parse(file_path)
        return True
    except ET.ParseError as e:
        print(f"XML parse error: {e}")
        return False

# Non-fatal: so you still reach the final cell and have xml_path / row_tag for Notebook 2
xml_well_formed = is_valid_xml(xml_path)
if xml_well_formed:
    print("✅ XML is well-formed")
else:
    print("❌ Malformed XML — use Notebook 2 with Run healer = true.")

**At the end of this notebook you know:** root tag, row tag, columns, nested structures, messiness level.

**Next:** Run **02_bronze_ingest_xml.ipynb** with the same `xml_path` and `row_tag`. If you saw malformed empty tags (e.g. `<>0</>`), set **Run healer** = true there.

In [0]:
# Copy these into 02_bronze_ingest_xml.ipynb widgets if running as a job
print(f"xml_path: {xml_path}")
print(f"row_tag:  {row_tag}")